# Lesson 28 Lab — GPU Memory, Concurrency, and Cost Estimation

**Puzzle:** How many requests fit after INT4 weight compression, and which hidden assumptions can invalidate that number?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

Cloud cost begins with a memory feasibility ledger, but it cannot end there. Ideal weight bits, unquantized layers, scale metadata, KV cache per request, workspace, fragmentation, tensor parallelism, throughput, utilization, and hourly price all determine whether one GPU is usable and economical.


## 0. Predict before running

1. Estimate ideal BF16 and INT4 weight GiB for 70B parameters.
2. Compute one-request KV cache for 80 layers, 8 KV heads, dimension 128, and 8K context.
3. Predict whether ideal INT4 weights fit a 32,607 MiB RTX 5090 after a 10% reserve.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

Capacity uses total/usable HBM, weight and scale bytes, runtime reserve, workspaces, KV per request, fragmentation, tensor parallelism, and traffic context distribution.

- Capacity starts from usable memory after runtime reserve, weights, workspaces, and fragmentation allowance.
- Per-request KV cache depends on context and cache dtype.
- Cost per token also depends on achieved throughput and utilization, not GPU price alone.


## 2. Derive the mechanism

A first bound is `requests = floor((usable - weights - workspace) / KV_per_request)`. Cost per token then depends on hourly price divided by achieved, quality-approved tokens per hour.

Weight bytes start at `P·bits/8`. KV bytes per request are `2·L·S·Hkv·D·cache_bytes`, then concurrency multiplies that term. A safety reserve should cover kernels, graph capture, allocator behavior, and unexpected peaks before dividing remaining bytes by per-request cache.

Even a memory fit does not produce a cost result. Cost per million tokens depends on achieved tokens/s, utilization, batching, power/cloud price, failure rate, and replica count. The notebook intentionally stops at arithmetic capacity when no engine throughput exists.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "28-gpu-capacity-cost"
device = require_cuda()
torch.manual_seed(2026 + 28)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | 70B BF16 weights with BF16 KV cache |
| Candidate | ideal INT4 weights with BF16 or INT8 KV cache |
| Held constant | 70B parameters, 80 layers, 8 KV heads, head dimension 128, context 8192, 10% reserve |
| Measurements | live total/free GiB, weight GiB, KV GiB/request, fit boolean, projected request count |
| Evidence | `capacity-model` |

**Experiment:** Read live free memory from the RTX GPU and build BF16 versus INT4 capacity projections for a 70B-class model without allocating the model.


## 5. Read the experiment code

The lab seeds a 70B arithmetic model with live RTX 5090 memory but explicitly does not allocate or benchmark a 70B model.

The notebook reads live RTX 5090 memory, calculates three plans, reserves 10%, and only then computes request capacity. It records zero rather than a negative or optimistic concurrency when weights already exceed usable memory.

The INT4 term is explicitly ideal: it excludes scales, padding, embeddings/norms retained in higher precision, engine, and workspace. That label prevents the arithmetic from being mistaken for a successful model load.

Only after these variables match the protocol should the cell be executed.


In [2]:
free,total=torch.cuda.mem_get_info(); cfg={"parameters":70_000_000_000,"layers":80,"kv_heads":8,"head_dim":128,"context":8192}
def plan(weight_bytes,cache_bytes):
    weights=cfg["parameters"]*weight_bytes; reserve=total*0.10; usable=max(0,total-reserve-weights); kv=2*cfg["layers"]*cfg["context"]*cfg["kv_heads"]*cfg["head_dim"]*cache_bytes
    return {"weight_gib":round(weights/2**30,3),"kv_per_request_gib":round(kv/2**30,3),"projected_requests":max(0,int(usable//kv)),"single_gpu_weight_fit":weights<total-reserve}
plans={"bf16_weights_bf16_kv":plan(2,2),"int4_ideal_weights_bf16_kv":plan(0.5,2),"int4_ideal_weights_int8_kv":plan(0.5,1)}
result=base_result(28,"capacity-model"); result.update({"live_memory":{"free_gib":round(free/2**30,3),"total_gib":round(total/2**30,3)},"assumptions":cfg,"plans":plans,
    "conclusion":"Arithmetic capacity projections used live GPU memory but did not claim that a 70B engine loaded or met latency SLOs."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Live total memory | 31.358 GiB |
| BF16 weight projection | 130.385 GiB |
| Ideal INT4 weight projection | 32.596 GiB |
| BF16 KV per request | 2.500 GiB |
| INT8 KV per request | 1.250 GiB |
| Ideal INT4 single-GPU fit | no |


## 7. Interpret rather than merely print

Live total memory was 31.358 GiB. BF16 weights projected to 130.385 GiB; ideal INT4 still required 32.596 GiB, already larger than total memory and larger still relative to the 10% reserve. BF16 KV cache was 2.5 GiB/request and INT8 KV 1.25 GiB/request, but every single-GPU plan correctly returned zero requests because weights did not fit.

KV compression cannot rescue a base model that fails the weight-fit gate. A real 70B deployment therefore needs further compression/overhead reduction, multi-GPU sharding, CPU offload, or a different GPU class before concurrency is discussed.

**Inspection rule:** Label the result as a capacity model. It cannot establish latency, model quality, or whether a particular 70B engine will load.


## 8. Keep the evidence label honest

This run is labeled **`capacity-model`**. The calculation uses live GPU information and/or a CUDA probe, but it remains a planning model until a named full engine, quality suite, and service workload execute.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "assumptions": {
    "context": 8192,
    "head_dim": 128,
    "kv_heads": 8,
    "layers": 80,
    "parameters": 70000000000
  },
  "conclusion": "Arithmetic capacity projections used live GPU memory but did not claim that a 70B engine loaded or met latency SLOs.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "capacity-model",
  "executed_at_utc": "2026-08-07T14:46:23+00:00",
  "lesson": 28,
  "live_memory": {
    "free_gib": 30.863,
    "total_gib": 31.358
  },
  "plans": {
    "bf16_weights_bf16_kv": {
      "kv_per_request_gib": 2.5,
      "projected_requests": 0,
      "single_gpu_weight_fit": false,
      "weight_gib": 130.385
    },
    "int4_ideal_weights_bf16_kv": {
      "kv_per_request_gib": 2.5,
      "projected_requests": 0,
      "single_gpu_weight_fit": false,
      "weight_gib": 32.596
    },


## 9. Make the bounded decision

> Use ranges and safety margins, then validate the chosen point with the actual engine and traffic distribution.

**Acceptance/rollback:** Use ranges and safety margins, then validate with the actual engine's measured peak, sustained concurrency, SLO, utilization, and cloud billing unit.

**Failure analysis:** Using decimal GB instead of binary GiB can create misleading margin near capacity. Ideal four-bit arithmetic omits metadata and high-precision tensors, and free memory on an otherwise empty process is not engine capacity. Cost comparisons without throughput and quality at equal SLO are also meaningless.


## 10. Extend the evidence

Add measured overhead from a real engine, tensor-parallel sharding/communication, fragmentation, and batch-dependent workspaces. Once the model loads, benchmark sustained tokens/s and compute cost per million tokens at equal quality and p95 latency across candidate GPU plans.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
